<p style="color:red; text-align:center; font-size:36px;">
  Testing Heteroskedasticity
</p>


In [ ]:
# Load the wooldridge package which contains all datasets required for this notebook
library(wooldridge)

**Problem 1**

Consider the following model to explain sleeping behavior:

$$
sleep = \beta_0 + \beta_1 totwrk + \beta_2 educ + \beta_3 age + \beta_4 age^2 + \beta_5 yngkid + \beta_6 male + u.
$$

(i) Write down a model that allows the variance of $u$ to differ between men and women. The variance should not depend on other factors.  
(ii) Use the data in **SLEEP75** to estimate the parameters of the model for heteroskedasticity. (You have to estimate the *sleep* equation by OLS, first, to obtain the OLS residuals.) Is the estimated variance of $u$ higher for men or for women?  
(iii) Is the variance of $u$ statistically different for men and for women?


**(i)** Given the equation

$$
sleep = \beta_0 + \beta_1 totwrk + \beta_2 educ + \beta_3 age + \beta_4 age^2 + \beta_5 yngkid + \beta_6 male + u,
$$

the assumption that the variance of $u$ given all explanatory variables depends only on gender is

$$
\text{Var}(u \mid totwrk, educ, age, yngkid, male) = \text{Var}(u \mid male) = \delta_0 + \delta_1 male
$$

Then the variance for women is simply $\delta_0$ and that for men is $\delta_0 + \delta_1$; the difference in variances is $\delta_1$.

**(ii)**

In [ ]:
# Load the sleep75 dataset
data("sleep75")

# View the first few rows of the sleep75 dataset
head(sleep75)

In [ ]:
# Run OLS regression
ols_model <- lm(sleep ~ totwrk + educ + age + I(age^2) + yngkid + male, data = sleep75)

summary(ols_model)

In [ ]:
# Get residuals
sleep75$resid <- resid(ols_model)

In [ ]:
# Compute residual variance by gender
var_female <- var(sleep75$resid[sleep75$male == 0])
var_male   <- var(sleep75$resid[sleep75$male == 1])

# Print the results
cat("Estimated variance of u for women:", round(var_female, 2), "\n")
cat("Estimated variance of u for men:  ", round(var_male, 2), "\n")

**(iii)**

We already estimated the residual variance for men and women in part (ii). Now, we test:

$$
H_0: \text{Var}(u \mid male = 0) = \text{Var}(u \mid male = 1)
$$


In [ ]:
# Perform F-test for equality of residual variances between men and women
resid_female <- sleep75$resid[sleep75$male == 0]
resid_male   <- sleep75$resid[sleep75$male == 1]

var_test <- var.test(resid_male, resid_female)

# Print test result
print(var_test)

In [ ]:
var_test <- var.test(resid_female, resid_male)

# Print test result
print(var_test)

### Interpretation of F-Test for Equality of Variances

The null hypothesis is:
$$
H_0: \text{Var}(u \mid male = 0) = \text{Var}(u \mid male = 1)
$$

The F-statistic is 0.847 with a p-value of 0.1206. Since the p-value is greater than the standard significance level (e.g., 0.05), we **fail to reject the null hypothesis**.

This means there is **no statistically significant difference** in the variance of the error term $u$ between men and women.

The 95% confidence interval for the variance ratio is:
$$
[0.6847,\ 1.0446]
$$
Since this interval contains 1, it supports the conclusion that the variances are not significantly different.

**Conclusion:**  
There is no evidence that the residual variance differs by gender.


**Problem 2**

Use **VOTE1** for this exercise.

**(i)** Estimate a model with `voteA` as the dependent variable and `prtystrA`, `democA`, $\log(expendA)$, and $\log(expendB)$ as independent variables. Obtain the OLS residuals, $\hat{u}_i$, and regress these on all of the independent variables.  
Explain why you obtain $R^2 = 0$.

**(ii)** Now, compute the **Breusch-Pagan test** for heteroskedasticity. Use the $F$ statistic version and report the *p*-value.

**(iii)** Compute the special case of the **White test** for heteroskedasticity, again using the $F$ statistic form.  
How strong is the evidence for heteroskedasticity now?

In [ ]:
# Load the vote1 dataset
data("vote1")

# View the first few rows of the vote1 dataset
head(vote1)

**(i)**

In [ ]:
# Estimate the original model
model1 <- lm(voteA ~ prtystrA + democA + log(expendA) + log(expendB), data = vote1)

summary(model1)

In [ ]:
# Get OLS residuals
vote1$resid <- resid(model1)

# Regress residuals on the same regressors
resid_model <- lm(resid ~ prtystrA + democA + log(expendA) + log(expendB), data = vote1)

# Display R-squared
summary(resid_model)$r.squared

### 🧠 Explanation: Why is $R^2 = 0$?

By construction, OLS residuals are **orthogonal** to the regressors used in the original model:

$$
\sum_i \hat{u}_i X_{ij} = 0 \quad \text{for each regressor } X_{ij}
$$

So when you regress $\hat{u}_i$ on the same $X$'s, you are regressing a variable that is **uncorrelated** with all regressors — leading to **zero explained variation**.

Thus:

$$
R^2 = 0
$$

This is a fundamental property of OLS residuals.


**(ii)**

### Breusch-Pagan Test for Heteroskedasticity

We use the Breusch-Pagan test to examine whether the error variance depends on the independent variables.

The null hypothesis is:
$$
H_0: \text{Var}(u_i) = \sigma^2 \quad \text{(homoskedasticity)}
$$



In [ ]:
# Load required packages (Install package first if not installed)
library(lmtest)

In [ ]:
# Compute squared residuals as the dependent variable
vote1$resid2 <- vote1$resid^2

resid_model <- lm(resid2 ~ prtystrA + democA + log(expendA) + log(expendB), data = vote1)

In [ ]:
# Extract values for F-statistic
r2 <- summary(resid_model)$r.squared
n  <- nobs(resid_model)
k  <- length(coef(resid_model)) - 1  # exclude intercept

# Compute Breusch-Pagan F-statistic
F_stat <- (r2 / k) / ((1 - r2) / (n - k - 1))

# Compute p-value
p_val <- pf(F_stat, df1 = k, df2 = n - k - 1, lower.tail = FALSE)

# Print results
cat("Breusch-Pagan F-statistic:", round(F_stat, 4), "\n")
cat("p-value:", round(p_val, 4), "\n")

### Breusch–Pagan Test for Heteroskedasticity (F-statistic version)

We test the null hypothesis that the error variance is constant (homoskedasticity):

$$
H_0: \text{Var}(u_i \mid X_i) = \sigma^2
$$

Using the auxiliary regression of squared residuals $\hat{u}_i^2$ on the original regressors, we obtain:

- $F$-statistic: 2.33  
- $p$-value: 0.0581

**Interpretation:**

- At the 5% significance level, we **fail to reject** the null hypothesis.
- At the 10% level, the result is **marginally significant**.

This suggests **some mild evidence of heteroskedasticity**, but not strong enough to be conclusive at conventional levels (5%).



### Alternative: Using `bptest()` for Breusch–Pagan Test

As an alternative to computing the $F$-statistic manually via an auxiliary regression, we can use the built-in `bptest()` function from the `lmtest` package.

We set `studentize = FALSE` to run the **classic Breusch–Pagan test**, which tests for linear forms of heteroskedasticity.

```r
bp_test <- bptest(model1, studentize = FALSE)


In [ ]:
# Perform Breusch-Pagan test using F-statistic version
bp_test <- bptest(model1, studentize = FALSE)  # classic BP test

# Print result
print(bp_test)

### Breusch–Pagan Test Result

- **Test statistic (BP)**: 9.9195  
- **Degrees of freedom**: 4  
- **p-value**: 0.04181


### Interpretation:

The Breusch–Pagan test evaluates whether the variance of the regression errors depends on the independent variables.

- **Null hypothesis**: Homoskedasticity (constant error variance)
  $$
  H_0: \text{Var}(u_i \mid X_i) = \sigma^2
  $$
- **Alternative hypothesis**: Heteroskedasticity — the error variance varies systematically with regressors.

Since the **p-value is 0.0418**, which is **less than 0.05**, we **reject the null hypothesis** at the 5% significance level.


### Conclusion:

There is **statistically significant evidence of heteroskedasticity** in the model.  
It is advisable to use **robust (heteroskedasticity-consistent) standard errors** when making inference.


### Comparison of Two Breusch–Pagan Test Versions

We evaluate heteroskedasticity in the model:

$$
voteA = \beta_0 + \beta_1 prtystrA + \beta_2 democA + \beta_3 \log(expendA) + \beta_4 \log(expendB) + u
$$

We use two forms of the Breusch–Pagan test:

---

#### 1. **Manual F-Statistic Version** (Auxiliary Regression)

- **Auxiliary regression**: squared residuals on regressors
- **Test statistic**:  
  $$
  F = 2.3301
  $$
- **p-value**: 0.0581

**Interpretation**:  
We **fail to reject** the null at the 5% level (but it's close).  
There is **mild evidence** of heteroskedasticity.

---

#### 2. **Built-in `bptest()` (Chi-Squared Version)**

- **Function**: `bptest(model1, studentize = FALSE)`
- **Test statistic**:  
  $$
  BP = 9.9195 \quad \text{(Chi-squared with 4 d.f.)}
  $$
- **p-value**: 0.04181

**Interpretation**:  
We **reject** the null hypothesis at the 5% significance level.  
There is **statistically significant evidence** of heteroskedasticity.

---

### 🧠 Summary

| Version        | Test Statistic | Distribution | p-value | Decision (5%)         |
|----------------|----------------|--------------|---------|------------------------|
| Manual (F)     | 2.3301         | $F(4, n-k-1)$ | 0.0581  | Fail to reject         |
| `bptest()`     | 9.9195         | $\chi^2(4)$  | 0.0418  | **Reject** $H_0$       |

While both tests assess the same null hypothesis, the **Chi-squared version is slightly more sensitive** here.  
To be cautious, **use robust standard errors** to account for possible heteroskedasticity.


**(iii)**

### White Test for Heteroskedasticity

The **White test** is a general diagnostic for heteroskedasticity that does **not assume a specific functional form** of the variance.

Unlike the Breusch–Pagan test, which assumes that the variance of the error term depends **linearly** on the regressors, the White test allows for:

- **Nonlinear relationships** (squares of regressors)
- **Interaction terms** (cross-products)

---

### Null hypothesis:
$$
H_0: \text{Var}(u_i \mid X_i) = \sigma^2 \quad \text{(homoskedasticity)}
$$

### Alternative hypothesis:
$$
H_1: \text{Var}(u_i \mid X_i) \text{ is a general function of the regressors (possibly nonlinear)}
$$

We compute the **F-statistic version** by:

1. Estimating the original model and obtaining residuals.
2. Regressing the squared residuals $\hat{u}_i^2$ on:
   - The original regressors
   - Their **squares**
   - Their **interaction terms**
3. Using the $R^2$ from this auxiliary regression to compute the $F$-statistic.


In [ ]:
# Step 2: Compute squared residuals
vote1$uhat2 <- resid(model1)^2

In [ ]:
# Create regressors needed for the White auxiliary regression
vote1$logA   <- log(vote1$expendA)
vote1$logB   <- log(vote1$expendB)
vote1$logA2  <- vote1$logA^2
vote1$logB2  <- vote1$logB^2
vote1$logAB  <- vote1$logA * vote1$logB


In [ ]:
# Auxiliary regression (White test)
white_model <- lm(uhat2 ~ prtystrA + democA + logA + logB + logA2 + logB2 + logAB, data = vote1)

summary(white_model)


In [ ]:
# Compute F-statistic
r2_white <- summary(white_model)$r.squared
n <- nobs(white_model)
k <- length(coef(white_model)) - 1  # exclude intercept

F_stat_white <- (r2_white / k) / ((1 - r2_white) / (n - k - 1))
p_val_white <- pf(F_stat_white, df1 = k, df2 = n - k - 1, lower.tail = FALSE)

# Step 6: Print results
cat("White test F-statistic:", round(F_stat_white, 4), "\n")
cat("p-value:", round(p_val_white, 4), "\n")



### Results:

- $F$-statistic = 3.0085  
- p-value = 0.0053

Since the **p-value is well below 0.01**, we **strongly reject** the null hypothesis.



### Conclusion:

There is **strong and statistically significant evidence of heteroskedasticity** in the model.  
You should use **robust (heteroskedasticity-consistent) standard errors** for valid inference.


### Why Does the White Test Detect Heteroskedasticity More Strongly Than the Breusch–Pagan Test?

The discrepancy arises because the two tests are designed to detect **different kinds of heteroskedasticity**.



### ⚖️ Breusch–Pagan (BP) Test

- Tests whether the **variance of the error term** depends **linearly** on the regressors:
  $$
  \text{Var}(u_i \mid X_i) = \delta_0 + \delta_1 X_{1i} + \dots + \delta_k X_{ki}
  $$

- Auxiliary regression includes only the **original regressors**.

- **More restrictive** — only picks up simple, linear forms of heteroskedasticity.

- ✅ Good at detecting straightforward patterns  
- ❌ May miss nonlinear or interaction-driven heteroskedasticity



### ⚡ White Test

- Much more general:
  - Includes:
    - Original regressors
    - **Squares** of regressors
    - **Interaction terms** (cross-products)

- Tests whether the error variance depends **in any nonlinear way** on the regressors:
  $$
  \text{Var}(u_i \mid X_i) = f(X_i, X_i^2, X_i X_j)
  $$

- ✅ Can detect complex heteroskedasticity (nonlinear, multiplicative)
- ❌ Slightly higher chance of false positives in small samples



### 🎓 Conclusion

The **White test** is more flexible and powerful because it allows for **nonlinear effects** and **inter**
